**## Week 1 — Foundations: Cleaning & Exploring the Applicant Dataset**
**Intern:** Ahmed
**Program:** NEDUET — CS&IT

In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("/content/sample_data/applicants.csv")

In [3]:
df.head()

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Briana Murray,briana.murray948@example.com,0340-7482175,Digital Marketing,Karachi University,10/16/2025,Selected
1,Stephanie Martin,stephanie.martin371@example.com,0364-7468723,Data Science,FAST-NUCES,16-Jul-2025,Under Review
2,Charles Schmidt,CHARLES.SCHMIDT195@EXAMPLE.COM,03222196937,Mobile App Development,NEDUET,08-Feb-2026,Under Review
3,Audrey Young DDS,audrey.young.dds291@example.com,03586926179,Digital Marketing,Institute of Business Administration,05/12/2025,Under Review
4,Thomas Bradley,thomas.bradley100@example.com,03376724238,Mobile App Development,DHA Suffa University,04/04/2026,Under Review


In [4]:
df.shape

(95, 7)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Applicant Name    92 non-null     object
 1   Email             87 non-null     object
 2   Phone             95 non-null     object
 3   Domain Applied    95 non-null     object
 4   University        91 non-null     object
 5   Application Date  95 non-null     object
 6   Status            92 non-null     object
dtypes: object(7)
memory usage: 5.3+ KB


In [6]:
df.isnull().sum()

,0
Applicant Name,3
Email,8
Phone,0
Domain Applied,0
University,4
Application Date,0
Status,3


In [7]:
df.duplicated().sum()

np.int64(3)

In [8]:
df.columns

Index(['Applicant Name', 'Email', 'Phone', 'Domain Applied', 'University',
       'Application Date', 'Status'],
      dtype='object')

In [9]:
before_rows=df.shape[0]
df=df.dropna(subset=["Applicant Name"])

In [10]:
print(f"Rows before: {before_rows}, rows after dropping missing names: {df.shape[0]}")

Rows before: 95, rows after dropping missing names: 92


## **HANDELING MISSING DATA**

In [11]:
df["Email"]=df["Email"].fillna("notprovided@gmail.com")

**Checking Email column for missing value**

In [12]:
df["Email"].isnull().sum()

np.int64(0)

In [13]:
df["University"]=df["University"].fillna("Not specified")
df["University"].isnull().sum()

np.int64(0)

In [14]:
df['Status'] = df['Status'].fillna('Under Review')
df['Status'].isnull().sum()

np.int64(0)

In [44]:
df.isnull().sum()

,0
Applicant Name,0
Email,0
Phone,0
Domain Applied,0
University,0
Application Date,0
Status,0


In [15]:
exact_dupl=df.duplicated().sum()
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows found: {exact_dupes}")
df = df.drop_duplicates()
print(f"Rows after removing exact duplicates: {df.shape[0]}")

Exact duplicate rows found: 3
Rows after removing exact duplicates: 89


In [16]:
# Normalize email for comparison (strip whitespace, lowercase) without overwriting the real column yet
email_key = df['Email'].str.strip().str.lower()
near_dupes = email_key.duplicated().sum()
print(f"Additional near-duplicate applications found (same email, different formatting elsewhere): {near_dupes}")

rows_before_email_dedupe = df.shape[0]
df = df.loc[~email_key.duplicated(keep='first')]
print(f"Rows after removing near-duplicates by Email: {df.shape[0]} (removed {rows_before_email_dedupe - df.shape[0]})")

Additional near-duplicate applications found (same email, different formatting elsewhere): 7
Rows after removing near-duplicates by Email: 82 (removed 7)


# **STANDARDIZATION**

In [17]:
text_cols = ['Applicant Name', 'Email', 'University', 'Domain Applied', 'Status']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

In [18]:
df['Applicant Name'] = df['Applicant Name'].str.title()
df['Email'] = df['Email'].str.lower()

In [19]:
df['Domain Applied'].str.strip().str.lower().unique()

array(['digital marketing', 'data science', 'mobile app development',
       'web development', 'webdev', 'app dev', 'mobile dev',
       'ui/ux design', 'mobile app dev', 'data-science', 'ds',
       'digi marketing'], dtype=object)

In [20]:
domain_mapping = {
    'web development': 'Web Development',
    'web dev': 'Web Development',
    'webdev': 'Web Development',
    'data science': 'Data Science',
    'data-science': 'Data Science',
    'ds': 'Data Science',
    'mobile app development': 'Mobile App Development',
    'mobile app dev': 'Mobile App Development',
    'mobile dev': 'Mobile App Development',
    'app dev': 'Mobile App Development',
    'ui/ux design': 'UI/UX Design',
    'ui ux design': 'UI/UX Design',
    'uiux design': 'UI/UX Design',
    'ux design': 'UI/UX Design',
    'digital marketing': 'Digital Marketing',
    'digi marketing': 'Digital Marketing',
    'marketing': 'Digital Marketing',
}

df['Domain Applied'] = df['Domain Applied'].str.strip().str.lower().map(domain_mapping)
df['Domain Applied'].unique()

array(['Digital Marketing', 'Data Science', 'Mobile App Development',
       'Web Development', 'UI/UX Design'], dtype=object)

In [21]:
assert df['Domain Applied'].isnull().sum() == 0, "Unmapped domain variation found — update domain_mapping!"
print("Unique domains:", sorted(df['Domain Applied'].unique()))
print("Count:", df['Domain Applied'].nunique())

Unique domains: ['Data Science', 'Digital Marketing', 'Mobile App Development', 'UI/UX Design', 'Web Development']
Count: 5


In [22]:
df['Status'] = df['Status'].str.strip().str.title()
df['Status'].unique()

array(['Selected', 'Under Review', 'Rejected'], dtype=object)

# **FIXED DATA TYPES**

In [23]:
df['Application Date'] = pd.to_datetime(df['Application Date'], errors='coerce', format='mixed', dayfirst=True)
df['Application Date'].isnull().sum()

np.int64(0)

In [24]:
df['Phone'] = df['Phone'].str.replace('-', '', regex=False).str.replace(' ', '', regex=False)
df['Phone'].head()

,Phone
0,03407482175
1,03647468723
2,03222196937
3,03586926179
4,03376724238


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 82 entries, 0 to 94
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    82 non-null     object        
 1   Email             82 non-null     object        
 2   Phone             82 non-null     object        
 3   Domain Applied    82 non-null     object        
 4   University        82 non-null     object        
 5   Application Date  82 non-null     datetime64[ns]
 6   Status            82 non-null     object        
dtypes: datetime64[ns](1), object(6)
memory usage: 5.1+ KB


In [26]:
print("Final shape:", df.shape)
print("\nRemaining nulls:\n", df.isnull().sum())
print("\nRemaining exact duplicates:", df.duplicated().sum())
print("\nDomain Applied categories:", df['Domain Applied'].nunique())
print("Application Date dtype:", df['Application Date'].dtype)

Final shape: (82, 7)

Remaining nulls:
 Applicant Name      0
Email               0
Phone               0
Domain Applied      0
University          0
Application Date    0
Status              0
dtype: int64

Remaining exact duplicates: 0

Domain Applied categories: 5
Application Date dtype: datetime64[ns]


# **EXPORTING CLEANED DATASET**

In [27]:
df.to_csv('applicants_cleaned.csv', index=False)
print("Saved applicants_cleaned.csv")

Saved applicants_cleaned.csv
